In [2]:
import torch
import numpy as np
import os

# Assume you have a PyTorch dataset
# from my_datasets import MyTorchDataset
# pytorch_dataset = MyTorchDataset()

# --- Create dummy data for demonstration ---
pytorch_dataset = [(torch.randn(3, 2240, 2240), i % 10) for i in range(10)]

# --- Preparation Step ---
sdir = '/projects/bdne/spandey3/FINAL_GOTHAM/GOTHAM/notebooks/tmp'

save_path = f"{sdir}/numpy_dataset"
os.makedirs(save_path, exist_ok=True)

file_list_path = f"{sdir}/numpy_file_list.txt"

with open(file_list_path, "w") as f:
    for i, (tensor, label) in enumerate(pytorch_dataset):
        filename = os.path.join(save_path, f"sample_{i:05d}.npy")
        np.save(filename, tensor.numpy())
        # DALI needs a file list mapping files to labels
        f.write(f"{os.path.basename(filename)} {label}\n")

print("Data saved in .npy format.")




Data saved in .npy format.


In [5]:
from nvidia.dali.pipeline import Pipeline
import nvidia.dali.fn as fn

class NumpyPipeline(Pipeline):
    def __init__(self, batch_size, num_threads, device_id):
        super(NumpyPipeline, self).__init__(batch_size, num_threads, device_id)
        # Point DALI to the file list and the root directory
        self.input = fn.readers.numpy(
            file_root="data/numpy_dataset",
            file_list="data/numpy_file_list.txt",
            # DALI will automatically shard the data across multiple GPUs
            shard_id=self.shard_id,
            num_shards=self.num_shards,
            # Read the whole dataset again if we reach the end
            stick_to_shard=False,
            shuffle_after_epoch=True
        )

    def define_graph(self):
        # The reader outputs the numpy array and its label
        tensors, labels = self.input()
        # Move tensor to GPU memory
        tensors_gpu = tensors.gpu()
        return tensors_gpu, labels


